In [19]:
!pip install optuna xlstm

# Shallow xLSTM Benchmark with Optuna & Symlog

In [20]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna

from xlstm import (
    xLSTMBlockStack,
    xLSTMBlockStackConfig,
    sLSTMBlockConfig,
    mLSTMBlockConfig,
    sLSTMLayerConfig,
    mLSTMLayerConfig
)

SEED = 42
DATA_PATH = Path("Merged_Dataset_yoy.csv")
PREDICTION_TARGETS = ["EBITDA", "Net_Income", "ROA"]
DEFAULT_LOOKBACK_DAYS = 365

MAX_EPOCHS = 70
PATIENCE = 10
TRAIN_YEAR_CUTOFF = 2019
VALID_YEAR_CUTOFF = 2021
OPTUNA_TRIALS = 70

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads() // 2))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## Data Loading - Improved Pipeline

In [21]:
def load_and_prepare_yoy_data(data_path: Path, lookback_days: int = 365):
    try:
        df = pd.read_csv(data_path)
    except FileNotFoundError:
        df = pd.read_csv("Merged_Dataset_yoy.csv")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Company", "Date"]).reset_index(drop=True)
    df["year"] = df["Date"].dt.year

    df["has_targets"] = df[PREDICTION_TARGETS].notna().any(axis=1)
    year_end_records = df[df["has_targets"]].copy()

    exclude_cols = {"Date", "Company", "year", "has_targets"} | set(PREDICTION_TARGETS)
    feature_cols = [col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

    sequences = []
    for company in df["Company"].unique():
        company_records = year_end_records[year_end_records["Company"] == company].copy()
        company_data = df[df["Company"] == company].copy()

        for _, year_end_row in company_records.iterrows():
            year = int(year_end_row["year"])
            year_end_date = year_end_row["Date"]

            prior_year_records = year_end_records[
                (year_end_records["Company"] == company) & (year_end_records["year"] == year - 1)
            ]
            if prior_year_records.empty:
                continue

            prior_row = prior_year_records.iloc[0]
            historical_data = company_data[company_data["Date"] <= year_end_date]
            if len(historical_data) < lookback_days:
                continue

            window = historical_data.tail(lookback_days).copy()

            window[feature_cols] = window[feature_cols].ffill().bfill()
            if window[feature_cols].isna().any().any():
                continue

            sequences.append({
                "company": company, "year": year, "year_end_date": year_end_date,
                "window_data": window[feature_cols].to_numpy(dtype=np.float32),
                **{f"current_{t.lower()}": year_end_row[t] for t in PREDICTION_TARGETS},
                **{f"prior_{t.lower()}": prior_row[t] for t in PREDICTION_TARGETS},
            })

    data_records = []
    for seq in sequences:
        for target_name in PREDICTION_TARGETS:
            current_val = seq[f"current_{target_name.lower()}"]
            prior_val = seq[f"prior_{target_name.lower()}"]
            if pd.isna(current_val) or pd.isna(prior_val):
                continue
            label_value = (current_val - prior_val) / (np.abs(prior_val) + 1e-8)
            data_records.append({
                "company": seq["company"], "year": seq["year"], "year_end_date": seq["year_end_date"],
                "target": target_name, "label_value": label_value,
                "window_data": seq["window_data"]
            })

    return pd.DataFrame(data_records), feature_cols

def split_by_year(data_df: pd.DataFrame, train_cutoff: int, valid_cutoff: int):
    train_data = data_df[data_df["year"] <= train_cutoff].copy()
    val_data = data_df[(data_df["year"] > train_cutoff) & (data_df["year"] <= valid_cutoff)].copy()
    test_data = data_df[data_df["year"] > valid_cutoff].copy()
    return train_data, val_data, test_data

data_df, feature_cols = load_and_prepare_yoy_data(DATA_PATH, lookback_days=DEFAULT_LOOKBACK_DAYS)

# --- LOCK IN BEST PIPELINE: SYMLOG & ROBUST SCALER ---
improved_data_df = data_df.copy()
idx_value = improved_data_df["target"].isin(PREDICTION_TARGETS)
y_val = improved_data_df.loc[idx_value, "label_value"]
improved_data_df.loc[idx_value, "label_value"] = np.sign(y_val) * np.log1p(np.abs(y_val))

train_data_imp, val_data_imp, test_data_imp = split_by_year(improved_data_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

selected_indices = [feature_cols.index(f) for f in feature_cols]
robust_scaler = RobustScaler()
train_windows_imp = np.vstack([row[:, selected_indices] for row in train_data_imp["window_data"]])
robust_scaler.fit(train_windows_imp)
print(f"Data Processed. Training sets: {len(train_data_imp)}")

Data Processed. Training sets: 3168


## Dataset and Multi-Task Preparation

In [22]:
def extract_mtl_df(df: pd.DataFrame) -> pd.DataFrame:
    mtl_records = []
    for (company, year), group in df.groupby(["company", "year"]):
        window_data = group.iloc[0]["window_data"]
        year_end_date = group.iloc[0]["year_end_date"]
        label_value = np.full(len(PREDICTION_TARGETS), np.nan, dtype=np.float32)
        for i, t in enumerate(PREDICTION_TARGETS):
            t_row = group[group["target"] == t]
            if not t_row.empty:
                label_value[i] = t_row.iloc[0]["label_value"]
        mtl_records.append({
            "company": company, "year": year, "year_end_date": year_end_date,
            "window_data": window_data, "label_value": label_value
        })
    return pd.DataFrame(mtl_records)

mtl_train_data = extract_mtl_df(train_data_imp)
mtl_val_data = extract_mtl_df(val_data_imp)
mtl_test_data = extract_mtl_df(test_data_imp)

class ImprovedYoYDatasetMTL(Dataset):
    def __init__(self, data_df, scaler, feature_indices):
        self.data_df = data_df.reset_index(drop=True)
        self.scaler = scaler
        self.feature_indices = feature_indices

    def __len__(self): return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]
        window = row["window_data"][:, self.feature_indices].copy()
        if self.scaler: window = self.scaler.transform(window)
        target = row["label_value"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return torch.from_numpy(window), torch.tensor(target_clean, dtype=torch.float32), torch.tensor(mask, dtype=torch.bool), len(window)

def collate_fn_improved_mtl(batch):
    batch.sort(key=lambda x: x[3], reverse=True)
    sequences = [x[0] for x in batch]
    targets = torch.stack([x[1] for x in batch])
    masks = torch.stack([x[2] for x in batch])
    lengths = torch.tensor([x[3] for x in batch])
    # xLSTM does not use torch packing natively. We feed it zero-padded batches.
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, masks, lengths

## xLSTM Model Architecture definition

Since `xLSTM` relies strictly on matrix processing, zero-padded `batch_first` tensors are used instead of `PyTorch pack_padded` utilities, taking care to extract only the actual final hidden state per element based on the masking lengths.

In [ ]:
class ImprovedShallow_xLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = len(PREDICTION_TARGETS), dropout: float = 0.2, block_type: str = 'mlstm'):
        super().__init__()
        self.proj = nn.Linear(input_size, hidden_size)
        slstm_config = sLSTMLayerConfig(backend='cuda')

        self.xlstm_config = xLSTMBlockStackConfig(
            slstm_block=sLSTMBlockConfig(slstm=slstm_config),
            mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig()),
            context_length=400, # Large enough to encompass max lookback sequence limits
            num_blocks=1, # Depth 1, strictly comparing to shallow LSTM
            embedding_dim=hidden_size,
            slstm_at=[0] if block_type == 'slstm' else []
        )

        self.xlstm = xLSTMBlockStack(self.xlstm_config)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size, num_targets)

    def forward(self, x, lengths):
        # x shape: [B, MaxSeq, input_size]
        B, S, _ = x.shape
        x_proj = self.proj(x)

        # Output strictly structured as [B, S, hidden_dim]
        out = self.xlstm(x_proj)

        # Dynamically recover the 'last valid hidden step' taking sequence padding into account
        idx = (lengths - 1).view(-1, 1).expand(-1, out.size(2)).unsqueeze(1).to(out.device)
        last_out = out.gather(1, idx).squeeze(1) # [B, hidden_size]

        hidden = self.dropout(last_out)
        logits = self.head(hidden)

        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

## Optuna & Training Loop Helper

In [ ]:
def train_model(model, train_loader, val_loader, criterion, lr, weight_decay, epochs, patience, device, trial=None):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_loss = float('inf')
    early_stop_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for x_batch, y_batch, mask_batch, lengths in train_loader:
            x_batch, y_batch, mask_batch, lengths = x_batch.to(device), y_batch.to(device), mask_batch.to(device), lengths.to(device)

            if mask_batch.sum() == 0:
                continue

            optimizer.zero_grad()
            preds = model(x_batch, lengths)
            loss = criterion(preds[mask_batch], y_batch[mask_batch])
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * x_batch.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch, mask_batch, lengths in val_loader:
                x_batch, y_batch, mask_batch, lengths = x_batch.to(device), y_batch.to(device), mask_batch.to(device), lengths.to(device)

                if mask_batch.sum() == 0:
                    continue

                preds = model(x_batch, lengths)
                loss = criterion(preds[mask_batch], y_batch[mask_batch])
                val_loss += loss.item() * x_batch.size(0)

        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)

        if trial is not None:
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if val_loss < best_loss:
            best_loss = val_loss
            early_stop_counter = 0
            torch.save(model.state_dict(), 'best_xlstm_model.pth')
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                break

    return best_loss

def evaluate_real_metrics(model, test_loader, device):
    model.eval()
    total_absolute_error = 0.0
    total_squared_error = 0.0
    total_samples = 0

    with torch.no_grad():
        for x_batch, y_batch_symlog, mask_batch, lengths in test_loader:
            x_batch, y_batch_symlog, mask_batch, lengths = x_batch.to(device), y_batch_symlog.to(device), mask_batch.to(device), lengths.to(device)
            if mask_batch.sum() == 0: continue

            preds_symlog = model(x_batch, lengths)
            valid_preds_symlog = preds_symlog[mask_batch]
            valid_y_symlog = y_batch_symlog[mask_batch]

            # Inverse Symlog transform back to the scale of human values
            preds_real = torch.sign(valid_preds_symlog) * torch.expm1(torch.abs(valid_preds_symlog))
            y_real = torch.sign(valid_y_symlog) * torch.expm1(torch.abs(valid_y_symlog))

            absolute_error = torch.abs(preds_real - y_real).sum().item()
            squared_error = torch.pow(preds_real - y_real, 2).sum().item()

            total_absolute_error += absolute_error
            total_squared_error += squared_error
            total_samples += valid_y_symlog.size(0)

    real_mae = total_absolute_error / total_samples
    real_rmse = (total_squared_error / total_samples) ** 0.5
    print(f"Test MAE  (Real Scale): {real_mae:.4f}")
    print(f"Test RMSE (Real Scale): {real_rmse:.4f}")
    print("="*50 + "\n")

    return real_mae, real_rmse

def objective(trial):
    # Hyperparams natively focused on xLSTM properties and Regularization
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    # xLSTM uniquely offers distinct recurrent engines (matrix vs scalar variants)
    block_type = trial.suggest_categorical('block_type', ['mlstm', 'slstm'])

    train_ds = ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices)
    val_ds = ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_improved_mtl)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_improved_mtl)

    model = ImprovedShallow_xLSTM(
        input_size=len(selected_indices),
        hidden_size=hidden_size,
        num_targets=len(PREDICTION_TARGETS),
        dropout=dropout,
        block_type=block_type
    )

    # Locked onto our winner criteria from LSTM Tests:
    criterion = nn.L1Loss()

    val_loss = train_model(model, train_loader, val_loader, criterion, lr, weight_decay,
                           epochs=MAX_EPOCHS, patience=PATIENCE, device=DEVICE, trial=trial)

    return val_loss


In [25]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study.optimize(objective, n_trials=OPTUNA_TRIALS)

# print("Best trial:")
# print("  Value: ", study.best_trial.value)
# print("  Params: ")
# for key, value in study.best_trial.params.items():
#     print(f"    {key}: {value}")

[I 2026-05-29 15:38:57,362] A new study created in memory with name: no-name-2eaf5ce2-7d2b-4afb-afc1-7f15233094cb
[I 2026-05-29 15:40:53,813] Trial 0 finished with value: 0.7604212951660156 and parameters: {'hidden_size': 128, 'dropout': 0.17981235303058707, 'lr': 0.0001327248765394057, 'weight_decay': 0.00012195831373066691, 'batch_size': 64, 'block_type': 'mlstm'}. Best is trial 0 with value: 0.7604212951660156.
[I 2026-05-29 15:41:41,986] Trial 1 finished with value: 0.657059154510498 and parameters: {'hidden_size': 128, 'dropout': 0.31659738371905055, 'lr': 0.0013176650469044714, 'weight_decay': 1.2776006426806774e-05, 'batch_size': 128, 'block_type': 'mlstm'}. Best is trial 1 with value: 0.657059154510498.
[I 2026-05-29 15:42:34,664] Trial 2 finished with value: 0.6377630845705669 and parameters: {'hidden_size': 128, 'dropout': 0.4216932818611603, 'lr': 0.0012615873950952464, 'weight_decay': 0.0011015874034970036, 'batch_size': 64, 'block_type': 'mlstm'}. Best is trial 2 with valu

In [ ]:

print("OPTUNA BEST INITIALIZATION PARAMETERS")
print(f"Best Trial Validation Loss (Symlog Space): {study.best_trial.value:.4f}")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")
print("="*50 + "\n")

optimal_batch_size = study.best_trial.params["batch_size"]
test_dataset = ImprovedYoYDatasetMTL(mtl_test_data, robust_scaler, selected_indices)
test_loader = DataLoader(test_dataset, batch_size=optimal_batch_size, shuffle=False, collate_fn=collate_fn_improved_mtl)

final_model = ImprovedShallow_xLSTM(
    input_size=len(selected_indices),
    hidden_size=study.best_trial.params["hidden_size"],
    num_targets=len(PREDICTION_TARGETS),
    dropout=study.best_trial.params["dropout"],
    block_type=study.best_trial.params["block_type"]
)

print("Executing final training pass to lock-in optimal weights...")
train_loader_final = DataLoader(ImprovedYoYDatasetMTL(mtl_train_data, robust_scaler, selected_indices), batch_size=optimal_batch_size, shuffle=True, collate_fn=collate_fn_improved_mtl)
val_loader_final = DataLoader(ImprovedYoYDatasetMTL(mtl_val_data, robust_scaler, selected_indices), batch_size=optimal_batch_size, shuffle=False, collate_fn=collate_fn_improved_mtl)

_ = train_model(
    model=final_model,
    train_loader=train_loader_final,
    val_loader=val_loader_final,
    criterion=nn.L1Loss(),
    lr=study.best_trial.params["lr"],
    weight_decay=study.best_trial.params["weight_decay"],
    epochs=MAX_EPOCHS,
    patience=PATIENCE,
    device=DEVICE
)

final_model.load_state_dict(torch.torch.load('best_xlstm_model.pth', map_location=DEVICE))
real_mae, real_rmse = evaluate_real_metrics(final_model, test_loader, DEVICE)

OPTUNA BEST INITIALIZATION PARAMETERS
Best Trial Validation Loss (Symlog Space): 0.6031
  hidden_size: 128
  dropout: 0.3304317620207427
  lr: 0.0044111402568607
  weight_decay: 0.005453665611103645
  batch_size: 64
  block_type: mlstm

Executing final training pass to lock-in optimal weights...
Test MAE  (Real Scale): 1.5728
Test RMSE (Real Scale): 8.6150



In [ ]:
def evaluate_per_target_metrics(model, test_loader, device, target_names):
    model.eval()
    target_mae = {name: 0.0 for name in target_names}
    target_samples = {name: 0 for name in target_names}

    with torch.no_grad():
        for x_batch, y_batch_symlog, mask_batch, lengths in test_loader:
            x_batch, y_batch_symlog, mask_batch, lengths = x_batch.to(device), y_batch_symlog.to(device), mask_batch.to(device), lengths.to(device)

            preds_symlog = model(x_batch, lengths)

            for i, name in enumerate(target_names):
                valid_mask = mask_batch[:, i]
                if valid_mask.sum() == 0: continue

                y_true_sym = y_batch_symlog[valid_mask, i]
                y_pred_sym = preds_symlog[valid_mask, i]

                y_true_real = torch.sign(y_true_sym) * torch.expm1(torch.abs(y_true_sym))
                y_pred_real = torch.sign(y_pred_sym) * torch.expm1(torch.abs(y_pred_sym))

                mae = torch.abs(y_true_real - y_pred_real).sum().item()

                target_mae[name] += mae
                target_samples[name] += valid_mask.sum().item()

    print("--- MAE PER TARGET (REAL SCALE) ---")
    results = {}
    for name in target_names:
        final_mae = target_mae[name] / target_samples[name] if target_samples[name] > 0 else float('nan')
        results[name] = final_mae
        print(f"{name:12} MAE: {final_mae:.4f}")
    print("="*35)
    return results

per_target_results = evaluate_per_target_metrics(final_model, test_loader, DEVICE, PREDICTION_TARGETS)

--- MAE PER TARGET (REAL SCALE) ---
EBITDA       MAE: 1.5211
Net_Income   MAE: 1.6115
ROA          MAE: 1.5851
